# GateKeep · notebook 02 · questions and the first baseline

**Before Run All:** Add Input → your private dataset (the book PDF, `gatekeep_backup.zip`, `review.csv`, `handwritten.csv`) · Settings → **Internet: On** · Accelerator: **None** (no GPU needed here; it saves your GPU hours) · Add-ons → Secrets → turn on `OLLAMA_API_KEY`.

You can stop after any step; every model call is cached on disk. **Keep this notebook private**: it handles book text.

**Long steps (questions, baseline):** an interactive session is shut down after a while without activity and erases its files. Run steps 1-4 interactively, then use **Save Version → Save & Run All (Commit)** so the rest runs unattended and the output (including the backup zip) is saved.


## 1 · get the code

In [ ]:
import glob, os, shutil, subprocess
os.chdir("/kaggle/working")
if os.path.isdir("GateKeep"):
    subprocess.run(["git", "-C", "GateKeep", "pull"], check=True)
else:
    subprocess.run(["git", "clone", "https://github.com/RishabhCodezZz/GateKeep.git"], check=True)
os.chdir("GateKeep")
os.environ["PYTHONPATH"] = "src"   # so `!python -m gatekeep.cli` finds the package
print(subprocess.run(["git", "log", "--oneline", "-1"], capture_output=True, text=True).stdout)

In [ ]:
!pip -q install docling sentence-transformers faiss-cpu pymupdf pytest openai

In [ ]:
from kaggle_secrets import UserSecretsClient
os.environ["OLLAMA_API_KEY"] = UserSecretsClient().get_secret("OLLAMA_API_KEY")

## 2 · restore your backup (skips the 10-minute parse and keeps the cached model replies)

In [ ]:
import sys
sys.path.insert(0, "src")
from gatekeep.restore import restore

COLD_START_OK = False   # set True only if you really want to rebuild everything (costs ~11% of the monthly quota)
where = restore("/kaggle/input", ".")
if where:
    print("restored from", where)
    subprocess.run(["python", "-m", "gatekeep.cli", "purge_empty"], check=True)  # drop empty replies cached before the thinking fix
elif not COLD_START_OK:
    raise SystemExit("STOPPED: no backup found in /kaggle/input. Attach your backup (the dataset must contain the folders "
                     "data/, cache/, results/ or a gatekeep_backup.zip), or set COLD_START_OK = True to rebuild from scratch.")
else:
    print("no backup: cold start, the book is re-parsed and every model call is repeated")
for name in ("review.csv", "handwritten.csv"):
    found = glob.glob(f"/kaggle/input/**/{name}", recursive=True)
    if found and not os.path.exists(name):
        shutil.copy(found[0], name)
    print(name, "->", "found" if os.path.exists(name) else "MISSING (add it to your dataset)")

## 3 · chunks and chapters (about 2 min if restored)

In [ ]:
pdf = glob.glob("/kaggle/input/**/*.pdf", recursive=True)[0]
print("PDF:", pdf)

In [ ]:
!python -m gatekeep.cli prepare "{pdf}"

## 4 · does the new question filter agree with your 50 marks? (about 100 calls, ~2 min)
Expect: *patterns* remove about 25 of the 31 bad questions and lose none of the good ones; *patterns+judge* should remove more. Paste both lines back to Claude.

In [ ]:
!python -m gatekeep.cli check_filter

## 5 · regenerate the questions with the filter (~1,500 calls, about 30 min, ~2% of the monthly quota)

In [ ]:
!python -m gatekeep.cli questions

In [ ]:
!wc -l data/qa_*.jsonl

## 6 · convert your hand-written questions

In [ ]:
!python -m gatekeep.cli handwritten

## 7 · the first real result: V0 (no checks) vs V1 (every check done by Gemma)
About 330 questions per version. Paste the two result lines and the usage line back to Claude.

In [ ]:
!python -m gatekeep.cli variants V0,V1

In [ ]:
!python -m gatekeep.cli breakdown

## 8 · back up, then stop the session

In [ ]:
!cd /kaggle/working/GateKeep && zip -rq /kaggle/working/gatekeep_backup.zip data cache results && ls -lh /kaggle/working/gatekeep_backup.zip

Download `gatekeep_backup.zip` from the **Output** panel, keep it **outside the git folder** (it holds book text), then use the power icon to **Stop Session**.